In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Exemplos 5.2 – 5.10  •  Capítulo 5: Memória Cache
Monteiro, "Introdução à Organização de Computadores", 5ª ed.

Mapeamentos abordados:
  Ex 5.2 – 5.4   →  Mapeamento Direto
  Ex 5.5 – 5.7   →  Mapeamento Associativo Completo
  Ex 5.8 – 5.10  →  Mapeamento Associativo por Conjunto (k = 4)

Erros identificados no livro:
  Ex 5.8 – Tt calculado como 8.192 bits (correto: 45.056 bits → T ≈ 300 Kbits)
  Ex 5.9 – Livro usa L = 8K linhas (correto: 32KB/16B = 2K linhas)
           Livro imprime Tag(11)|Conj(11)|Byte(4); correto: Tag(13)|Conj(9)|Byte(4)
  Ex 5.10 – Enunciado menciona cache de 64 KB, mas a solução do livro usa 128 KB
"""

import math


# ═══════════════════════════════  utilitários  ════════════════════════════════

def log2i(n: int) -> int:
    """log₂ inteiro."""
    return int(math.log2(n))

def hex2bin(h: str) -> str:
    """'3FC92B6' → string binária de (4 × len) bits com zeros à esquerda."""
    return format(int(h, 16), f"0{len(h) * 4}b")

def split_bin(bits: str, *sizes) -> list:
    """Divide a string binária em campos de tamanhos dados (esquerda → direita)."""
    campos, pos = [], 0
    for s in sizes:
        campos.append(bits[pos : pos + s])
        pos += s
    return campos

def grp4(s: str) -> str:
    """Agrupa em blocos de 4 para leitura: '001011' → '0010 11'."""
    return " ".join(s[i : i + 4] for i in range(0, len(s), 4))

def titulo(t: str):
    bar = "═" * (len(t) + 6)
    print(f"\n╔{bar}╗")
    print(f"║   {t}   ║")
    print(f"╚{bar}╝")

def fmt_end(*campos):
    """
    Desenha a caixa de formato de endereço.
    Uso:  fmt_end(("Tag", 11), ("Linha", 11), ("Byte", 4))
    """
    total = sum(b for _, b in campos)
    W = [max(10, len(nm) + 4) for nm, _ in campos]

    def sep(c):
        return c.join("─" * w for w in W)

    print(f"\n  ← {total} bits →")
    print(f"  ┌{sep('┬')}┐")
    print(f"  │{'│'.join(nm.center(w) for (nm, _), w in zip(campos, W))}│")
    print(f"  └{sep('┴')}┘")
    # linha de tamanhos alinhada com os centros das colunas
    print("   " + "".join(f"{b} bits".center(w + 1) for (_, b), w in zip(campos, W)))


# ═══════════════════════════════  Exemplo 5.2  ════════════════════════════════
# Mapeamento Direto — cálculo do TOTAL DE BITS da cache

def ex52():
    titulo("5.2 — Mapeamento Direto — Total de bits da cache")
    print("  Dados: cache = 32 KB  |  linha/bloco = 8 B  |  MP = 16 MB\n")

    C = 32 * 1024           # capacidade da cache (bytes)
    X = 8                   # largura de linha/bloco (bytes)
    M = 16 * 1024 * 1024    # capacidade da MP (bytes)

    L   = C // X            # número de linhas da cache
    B   = M // X            # número de blocos da MP
    bpl = B // L            # blocos por linha → determina o campo Tag
    tag = log2i(bpl)        # bits de tag

    Td = C * 8              # bits de dados (linhas × bytes × 8)
    Tt = L * tag            # bits de tag   (linhas × largura do campo tag)
    T  = Td + Tt

    print(f"  L   = {C//1024} KB / {X} B      = {L//1024} K linhas    = 2^{log2i(L)}")
    print(f"  B   = {M//1024//1024} MB / {X} B      = {B//1024//1024} MB blocos   = 2^{log2i(B)}")
    print(f"  B/L = 2^{log2i(B)} / 2^{log2i(L)}    = 2^{tag} blocos/linha → Tag = {tag} bits")
    print(f"\n  Td  = {C//1024}K × 8     = {Td:,} bits   (dados)")
    print(f"  Tt  = {L//1024}K × {tag}      = {Tt:,} bits    (tags)")
    print(f"  T   = Td + Tt       = {T:,} bits ≈ {T // 1024} Kbits")


# ═══════════════════════════════  Exemplo 5.3  ════════════════════════════════
# Mapeamento Direto — FORMATO DO ENDEREÇO

def ex53():
    titulo("5.3 — Mapeamento Direto — Formato do endereço")
    print("  Dados: MP = 64 MB  |  L = 2 K linhas  |  largura = 16 B\n")

    M = 64 * 1024 * 1024
    L = 2 * 1024
    X = 16

    E    = log2i(M)         # bits totais do endereço da MP
    byte = log2i(X)         # bits do campo Byte
    lin  = log2i(L)         # bits do campo Linha
    B    = M // X           # total de blocos
    bpl  = B // L           # blocos por linha
    tag  = log2i(bpl)       # bits do campo Tag

    print(f"  E      = log₂({M//1024//1024} MB)  = {E} bits  (endereço completo da MP)")
    print(f"  B      = {M//1024//1024} MB / {X} B  = {B//1024//1024} MB = 2^{log2i(B)} blocos")
    print(f"  B/L    = 2^{log2i(B)} / 2^{lin}  = 2^{tag} blocos/linha  → Tag = {tag} bits")
    print(f"  Linha  = log₂({L//1024}K)  = {lin} bits")
    print(f"  Byte   = log₂({X})   = {byte} bits")
    print(f"  Cheque : {tag} + {lin} + {byte} = {tag+lin+byte} = E ✓")
    fmt_end(("Tag", tag), ("Linha", lin), ("Byte", byte))


# ═══════════════════════════════  Exemplo 5.4  ════════════════════════════════
# Mapeamento Direto — DECODIFICAÇÃO do endereço 0x3FC92B6

def ex54():
    titulo("5.4 — Mapeamento Direto — Decodificação de 0x3FC92B6")
    print("  Dados: bloco = 32 B  |  cache = 128 KB  |  endereço = 0x3FC92B6\n")

    HEX = "3FC92B6"
    X   = 32          # bytes por bloco/linha
    C   = 128 * 1024  # capacidade da cache

    nb   = len(HEX) * 4    # bits totais (7 dígitos hex = 28 bits)
    byte = log2i(X)        # campo Byte
    L    = C // X          # linhas da cache
    lin  = log2i(L)        # campo Linha
    tag  = nb - lin - byte # campo Tag

    b              = hex2bin(HEX)
    t_v, l_v, b_v = split_bin(b, tag, lin, byte)

    print(f"  0x{HEX}  =  {grp4(b)}  ({nb} bits)")
    print(f"  Byte    = log₂({X}) = {byte} bits")
    print(f"  L       = {C//1024} KB / {X} B  = {L//1024} K = 2^{lin}  → Linha = {lin} bits")
    print(f"  Tag     = {nb} − {lin} − {byte} = {tag} bits")
    fmt_end(("Tag", tag), ("Linha", lin), ("Byte", byte))
    print(f"\n  Tag   = {grp4(t_v)}")
    print(f"  Linha = {grp4(l_v)}  =  {int(l_v, 2):,}₁₀  →  linha #{int(l_v, 2)}")
    print(f"  Byte  = {grp4(b_v)}  = {int(b_v, 2)}₁₀")


# ═══════════════════════════════  Exemplo 5.5  ════════════════════════════════
# Mapeamento Associativo Completo — TOTAL DE BITS da cache

def ex55():
    titulo("5.5 — Mapeamento Associativo Completo — Total de bits da cache")
    print("  Dados: cache = 32 KB  |  linha = 8 B  |  MP = 16 MB\n")

    C = 32 * 1024
    X = 8
    M = 16 * 1024 * 1024

    L   = C // X
    B   = M // X
    blk = log2i(B)   # tag = endereço completo do bloco = log₂(B)
                     # no mapeamento associativo NÃO há campo Linha

    Td = C * 8
    Tb = L * blk
    T  = Td + Tb

    print(f"  L            = {C//1024} KB / {X} B  = {L//1024}K linhas  = 2^{log2i(L)}")
    print(f"  B            = {M//1024//1024} MB / {X} B  = {B//1024//1024}MB blocos  = 2^{blk}")
    print(f"  Campo Bloco  = log₂(B) = {blk} bits  (tag = endereço do bloco; sem campo Linha)")
    print(f"\n  Td  = {C//1024}K × 8    = {Td:,} bits")
    print(f"  Tb  = {L//1024}K × {blk}     = {Tb:,} bits")
    print(f"  T   = Td + Tb    = {T:,} bits ≈ {T // 1024} Kbits")


# ═══════════════════════════════  Exemplo 5.6  ════════════════════════════════
# Mapeamento Associativo Completo — FORMATO DO ENDEREÇO

def ex56():
    titulo("5.6 — Mapeamento Associativo Completo — Formato do endereço")
    print("  Dados: MP = 64 MB  |  L = 2 K linhas  |  largura = 16 B\n")

    M = 64 * 1024 * 1024
    X = 16

    E    = log2i(M)
    byte = log2i(X)
    B    = M // X
    blk  = log2i(B)   # sem campo Linha; tag = endereço do bloco

    print(f"  E          = log₂({M//1024//1024} MB)  = {E} bits")
    print(f"  B          = {M//1024//1024} MB / {X} B  = {B//1024//1024} MB = 2^{blk} blocos")
    print(f"  Bloco/Tag  = log₂(B) = {blk} bits  (sem campo Linha no mapeamento associativo)")
    print(f"  Byte       = log₂({X}) = {byte} bits")
    print(f"  Cheque : {blk} + {byte} = {blk+byte} = E ✓")
    fmt_end(("Bloco/Tag", blk), ("Byte", byte))


# ═══════════════════════════════  Exemplo 5.7  ════════════════════════════════
# Mapeamento Associativo Completo — DECODIFICAÇÃO de 0x3FC92B6

def ex57():
    titulo("5.7 — Mapeamento Associativo Completo — Decodificação de 0x3FC92B6")
    print("  Dados: bloco = 32 B  |  cache = 64 KB  |  endereço = 0x3FC92B6\n")

    HEX = "3FC92B6"
    X   = 32

    nb   = len(HEX) * 4
    byte = log2i(X)
    blk  = nb - byte   # não há campo Linha

    b           = hex2bin(HEX)
    bk_v, b_v  = split_bin(b, blk, byte)

    print(f"  0x{HEX}  =  {grp4(b)}  ({nb} bits)")
    print(f"  Byte   = log₂({X}) = {byte} bits")
    print(f"  B      = 2^{nb} / 2^{byte} = 2^{blk} = {2**blk // 1024 // 1024} M blocos  → Bloco = {blk} bits")
    fmt_end(("Bloco", blk), ("Byte", byte))
    print(f"\n  Bloco = {grp4(bk_v)}  (bloco #{int(bk_v, 2):,})")
    print(f"  Byte  = {grp4(b_v)} = {int(b_v, 2)}₁₀")


# ═══════════════════════════════  Exemplo 5.8  ════════════════════════════════
# Mapeamento Associativo por Conjunto k=4 — TOTAL DE BITS da cache

def ex58():
    titulo("5.8 — Assoc. por Conjunto k=4 — Total de bits da cache")
    print("  Dados: cache = 32 KB  |  linha = 8 B  |  k = 4  |  MP = 16 MB\n")

    C = 32 * 1024
    X = 8
    k = 4
    M = 16 * 1024 * 1024

    L   = C // X     # linhas da cache
    K   = L // k     # conjuntos
    B   = M // X     # blocos da MP
    bpc = B // K     # blocos por conjunto → campo Tag
    tag = log2i(bpc)

    Td = C * 8
    Tt = L * tag     # CORRETO: linhas × largura do campo tag
    T  = Td + Tt

    # Valor incorreto impresso pelo livro
    Tt_lv = 8_192
    T_lv  = Td + Tt_lv

    print(f"  L   = {C//1024} KB / {X} B    = {L//1024}K linhas     = 2^{log2i(L)}")
    print(f"  K   = {L//1024}K / {k}          = {K//1024}K conjuntos  = 2^{log2i(K)}")
    print(f"  B   = {M//1024//1024} MB / {X} B    = {B//1024//1024}MB blocos    = 2^{log2i(B)}")
    print(f"  B/K = 2^{log2i(B)} / 2^{log2i(K)}   = 2^{tag} blocos/conj  → Tag = {tag} bits")
    print(f"\n  Td  = {C//1024}K × 8    = {Td:,} bits")
    print(f"\n  ✅  CORRETO:")
    print(f"      Tt  = {L//1024}K linhas × {tag} bits = {Tt:,} bits")
    print(f"      T   = {Td:,} + {Tt:,}  = {T:,} bits ≈ {T // 1024} Kbits")
    print(f"\n  ❌  LIVRO (erro tipográfico):")
    print(f"      Livro imprime Tt = {Tt_lv:,} bits  →  T ≈ {T_lv // 1024} Kbits")
    print(f"      (escreveu '2K' em vez de '{tag} bits' para a largura do campo tag)")


# ═══════════════════════════════  Exemplo 5.9  ════════════════════════════════
# Mapeamento Associativo por Conjunto k=4 — FORMATO DO ENDEREÇO

def ex59():
    titulo("5.9 — Assoc. por Conjunto k=4 — Formato do endereço")
    print("  Dados: MP = 64 MB  |  cache = 32 KB  |  linha = 16 B  |  k = 4\n")

    M = 64 * 1024 * 1024
    C = 32 * 1024
    X = 16
    k = 4

    E    = log2i(M)
    byte = log2i(X)
    B    = M // X           # total de blocos da MP

    # ── Cálculo CORRETO ──────────────────────────────────────────────────────
    L_ok  = C // X          # 32KB / 16B = 2K  (livro imprime 8K — erro)
    K_ok  = L_ok // k       # 2K / 4 = 512 conjuntos
    tag_c = log2i(B // K_ok)   # 13 bits
    con_c = log2i(K_ok)        #  9 bits

    # ── Cálculo do LIVRO (errado) ──────────────────────────────────────────
    L_lv  = 8 * 1024        # livro calcula 32KB/16B = 8K  (deveria ser 2K)
    K_lv  = L_lv // k
    tag_l = log2i(B // K_lv)   # 11 bits
    con_l = log2i(K_lv)        # 11 bits

    print(f"  E = {E}  |  B = {B//1024//1024}M blocos = 2^{log2i(B)}  |  Byte = {byte} bits")

    print(f"\n  ✅  CORRETO  —  L = {C//1024}KB / {X}B = {L_ok//1024}K linhas:")
    print(f"      K   = {L_ok//1024}K / {k}  = {K_ok} conjuntos = 2^{con_c}")
    print(f"      B/K = 2^{log2i(B)} / 2^{con_c}  = 2^{tag_c}  → Tag = {tag_c} bits")
    print(f"      Cheque: {tag_c} + {con_c} + {byte} = {tag_c+con_c+byte} = E ✓")
    fmt_end(("Tag", tag_c), ("Conjunto", con_c), ("Byte", byte))

    print(f"\n  ❌  LIVRO  —  usa L = {L_lv//1024}K linhas (correto é {L_ok//1024}K):")
    print(f"      K   = {L_lv//1024}K / {k} = {K_lv//1024}K = 2^{con_l}  →  Tag={tag_l}, Conj={con_l}")
    fmt_end(("Tag", tag_l), ("Conjunto", con_l), ("Byte", byte))


# ═══════════════════════════════  Exemplo 5.10  ═══════════════════════════════
# Mapeamento Associativo por Conjunto k=4 — DECODIFICAÇÃO de 0x3FC92B6

def ex510():
    titulo("5.10 — Assoc. por Conjunto k=4 — Decodificação de 0x3FC92B6")
    print("  Dados: bloco = 32 B  |  cache = 128 KB*  |  k = 4  |  end = 0x3FC92B6")
    print("  (* enunciado menciona 64 KB; a solução do livro usa 128 KB)\n")

    HEX = "3FC92B6"
    X   = 32
    C   = 128 * 1024   # 128 KB conforme a solução do livro
    k   = 4

    nb   = len(HEX) * 4    # 28 bits
    byte = log2i(X)        # 5 bits
    L    = C // X          # linhas: 128K/32 = 4K
    K    = L // k          # conjuntos: 4K/4 = 1K
    con  = log2i(K)        # 10 bits
    tag  = nb - con - byte # 13 bits

    b              = hex2bin(HEX)
    t_v, c_v, b_v = split_bin(b, tag, con, byte)

    print(f"  0x{HEX}  =  {grp4(b)}  ({nb} bits)")
    print(f"  Byte     = log₂({X}) = {byte} bits")
    print(f"  L        = {C//1024}KB / {X}B  = {L//1024}K = 2^{log2i(L)}")
    print(f"  K        = {L//1024}K / {k}  = {K//1024}K conjuntos = 2^{con}")
    print(f"  Conjunto = {con} bits")
    print(f"  Tag      = {nb} − {con} − {byte} = {tag} bits")
    fmt_end(("Tag", tag), ("Conjunto", con), ("Byte", byte))
    print(f"\n  Tag      = {grp4(t_v)}")
    print(f"  Conjunto = {grp4(c_v)}  =  {int(c_v, 2):,}₁₀  →  conjunto #{int(c_v, 2)}")
    print(f"  Byte     = {grp4(b_v)} = {int(b_v, 2)}₁₀")


# ═══════════════════════════════════  main  ═══════════════════════════════════

if __name__ == "__main__":
    SEP = "═" * 66
    print(SEP)
    print("  Monteiro — Introdução à Organização de Computadores, 5ª ed.")
    print("  Capítulo 5: Memória Cache  ·  Exemplos 5.2 a 5.10")
    print(SEP)

    print("\n── MAPEAMENTO DIRETO ────────────────────────────────────────────")
    ex52(); ex53(); ex54()

    print("\n── MAPEAMENTO ASSOCIATIVO COMPLETO ──────────────────────────────")
    ex55(); ex56(); ex57()

    print("\n── MAPEAMENTO ASSOCIATIVO POR CONJUNTO (k = 4) ──────────────────")
    ex58(); ex59(); ex510()

    print(f"\n{SEP}\n  Concluído.\n{SEP}\n")
